# Github Repository Chroma Query

In [68]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
from collections import Counter
from datetime import datetime, timezone


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.github_document_loader import GitHubChromaDocumentLoader
from apps.agentic.core.constants import GITHUB_DB_NAME, GITHUB_COLLECTION_NAME
from apps.agentic.core.utils import (set_chatgpt_env, set_langsmith_env, set_tavily_env, 
                                     set_github_env)

DB_PATH = "../../.db"

set_chatgpt_env(filedir="../../.keys")
set_langsmith_env(filedir="../../.keys")

## Verify Document Counts

In [69]:
GITHUB_DB_NAME, GITHUB_COLLECTION_NAME, DB_PATH, os.listdir(DB_PATH)

('github',
 'github-repos',
 '../../.db',
 ['research_notes', '.DS_Store', 'chroma.sqlite3', 'github'])

In [70]:
vs = GitHubChromaDocumentLoader(DB_PATH).vectorstore
coll = vs._collection

In [71]:
client = vs._client
print([c.name for c in client.list_collections()])
print("Opened:", coll.name)

['github-repos']
Opened: github-repos


In [72]:
print("total docs:", coll.count())

total docs: 905


In [73]:
res = coll.get(where={"repo": "zgomot"})   # no 'include' arg at all
print("zgomot docs:", len(res["ids"]))

zgomot docs: 192


## Verify Document Metadata

In [74]:
probe = coll.get(limit=5000)  # adjust as needed
metas  = [m for m in probe.get("metadatas", []) if m]
len(metas)

905

In [75]:
keys = set().union(*(m.keys() for m in metas)) if metas else set()
print("metadata keys:", sorted(keys))


metadata keys: ['account', 'branch', 'commit', 'commit_ts', 'commit_ts_unix', 'ext', 'file_name', 'file_path', 'file_type', 'filename', 'path', 'repo', 'source', 'start_index']


In [76]:
print("repo counts:", Counter(m.get("repo") for m in metas if "repo" in m and m.get("repo")).most_common(10))
print("account counts:", Counter(m.get("account") for m in metas if "account" in m and m.get("account")).most_common(10))

repo counts: [('agent_xmpp', 604), ('zgomot', 192), ('data_examples', 109)]
account counts: [('troystribling', 905)]


## Search Validation

### Similarity

In [77]:
vs = GitHubChromaDocumentLoader(DB_PATH).vectorstore

docs = vs.similarity_search("README summary", k=5, filter={"repo": "zgomot"})
for d in docs:
    print(d.metadata.get("path"), "—", d.metadata.get("commit_ts"))
    print(d.page_content[:160].replace("\n"," "), "\n")

LICENSE — 2009-08-08T02:33:39+00:00
The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.  THE SOFTWARE IS PROVIDED "AS IS" 

LICENSE — 2009-08-08T02:33:39+00:00
The MIT License  Copyright (c) 2009 troystribling  Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated  

VERSION — 2015-03-07T19:32:16+00:00
1.1.0 

.document — 2009-08-07T23:35:44+00:00
README.rdoc lib/**/*.rb bin/* features/**/*.feature LICENSE 

lib/zgomot/comp/chord.rb — 2013-08-17T02:36:11+00:00
end  end 



### MMR Search

In [78]:
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 50, "filter": {"repo": "zgomot"}}
)

hits = retriever.invoke("Where is MIDI output handled?")
for d in hits[:5]:
    print(d.metadata.get("path"), d.metadata.get("commit_ts"))

lib/zgomot/drivers/core_midi.rb 2013-08-17T22:04:27+00:00
lib/zgomot/midi/note.rb 2013-08-17T02:36:11+00:00
README.rdoc 2015-03-07T20:58:50+00:00
README.rdoc 2015-03-07T20:58:50+00:00
README.rdoc 2015-03-07T20:58:50+00:00


## Search Examples 

In [79]:
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
        "filter": {"$and": [{"repo": "zgomot"}, {"ext": ".rb"}, {"account": "troystribling"}]},
    },
)
hits = retriever.invoke("Where is MIDI output handled? (drivers, note_on, note_off, port, send)")

[print(h) for h in hits]

page_content='class Zgomot::Drivers

  class CoreMidi < Driver

    attr_reader :destinations, :sources, :input, :output

    # API Wrapper
    module Interface

      extend FFI::Library
      ffi_lib '/System/Library/Frameworks/CoreMIDI.framework/Versions/Current/CoreMIDI'

      typedef :pointer, :CFStringRef
      typedef :int32,   :ItemCount
      typedef :pointer, :MIDIClientRef
      typedef :pointer, :MIDIDeviceRef
      typedef :pointer, :MIDIEntityRef
      typedef :pointer, :MIDIObjectRef
      typedef :pointer, :MIDIEndpointRef
      typedef :uint32,  :MIDITimeStamp
      typedef :pointer, :MIDIPortRef
      typedef :int32,   :OSStatus

      class MIDIPacket < FFI::Struct
        layout :timestamp,  :MIDITimeStamp,
               :nothing,    :uint32,
               :length,     :uint16,
               :data,       [:uint8, 256]
      end

      class MIDIPacketList < FFI::Struct
        layout :numPackets,   :uint32,
               :packet,       [MIDIPacket.by_value, 1]


[None, None, None, None, None, None, None, None]

### Use MMR Call Directly

In [60]:
docs = vs.max_marginal_relevance_search(
    "Where is MIDI output handled? (drivers, note_on, note_off, port, send)",
    k=8,
    fetch_k=60,
    filter={"$and": [{"repo": "zgomot"}, {"ext": ".rb"}]},
)

[print(h) for h in hits]

page_content='class Zgomot::Drivers

  class CoreMidi < Driver

    attr_reader :destinations, :sources, :input, :output

    # API Wrapper
    module Interface

      extend FFI::Library
      ffi_lib '/System/Library/Frameworks/CoreMIDI.framework/Versions/Current/CoreMIDI'

      typedef :pointer, :CFStringRef
      typedef :int32,   :ItemCount
      typedef :pointer, :MIDIClientRef
      typedef :pointer, :MIDIDeviceRef
      typedef :pointer, :MIDIEntityRef
      typedef :pointer, :MIDIObjectRef
      typedef :pointer, :MIDIEndpointRef
      typedef :uint32,  :MIDITimeStamp
      typedef :pointer, :MIDIPortRef
      typedef :int32,   :OSStatus

      class MIDIPacket < FFI::Struct
        layout :timestamp,  :MIDITimeStamp,
               :nothing,    :uint32,
               :length,     :uint16,
               :data,       [:uint8, 256]
      end

      class MIDIPacketList < FFI::Struct
        layout :numPackets,   :uint32,
               :packet,       [MIDIPacket.by_value, 1]


[None, None, None, None, None, None, None, None]

In [61]:
after = datetime(2014,1,1, tzinfo=timezone.utc).timestamp()

flt = {
  "$and": [
    {"repo": "zgomot"},
    {"ext": ".rb"},
    {"commit_ts": {"$gte": after}}
  ]
}
docs = vs.similarity_search("note_on handler", k=8, filter=flt)
[print(h) for h in hits]

page_content='class Zgomot::Drivers

  class CoreMidi < Driver

    attr_reader :destinations, :sources, :input, :output

    # API Wrapper
    module Interface

      extend FFI::Library
      ffi_lib '/System/Library/Frameworks/CoreMIDI.framework/Versions/Current/CoreMIDI'

      typedef :pointer, :CFStringRef
      typedef :int32,   :ItemCount
      typedef :pointer, :MIDIClientRef
      typedef :pointer, :MIDIDeviceRef
      typedef :pointer, :MIDIEntityRef
      typedef :pointer, :MIDIObjectRef
      typedef :pointer, :MIDIEndpointRef
      typedef :uint32,  :MIDITimeStamp
      typedef :pointer, :MIDIPortRef
      typedef :int32,   :OSStatus

      class MIDIPacket < FFI::Struct
        layout :timestamp,  :MIDITimeStamp,
               :nothing,    :uint32,
               :length,     :uint16,
               :data,       [:uint8, 256]
      end

      class MIDIPacketList < FFI::Struct
        layout :numPackets,   :uint32,
               :packet,       [MIDIPacket.by_value, 1]


[None, None, None, None, None, None, None, None]